# Explore the maths RAG knowledge base

This notebook lets you poke at `data/processed/questions.db` directly — no LLM involved yet, just checking that the retrieval mechanics (tags → questions → techniques) work the way we designed them.

Run each cell top to bottom with **Shift+Enter**. If you edit any JSON under `data/processed/` later (questions, topics, techniques, examiner notes or `tags.json`), re-run `python3 scripts/build_db.py` from a terminal first, then re-run the cells here to see the changes.

In [1]:
import sqlite3
from pathlib import Path

DB_PATH = Path.cwd().parent / "data" / "processed" / "questions.db"
assert DB_PATH.exists(), f"Database not found at {DB_PATH} — run scripts/build_db.py first"

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
print(f"Connected to {DB_PATH}")

Connected to /Users/i1003721/edexcel-maths/data/processed/questions.db


## 1. What's in the knowledge base right now?

A quick tour of every table.

In [2]:
print("Questions:")
for r in conn.execute("SELECT id, paper, sitting, q_num, total_marks FROM questions"):
    print(f"  {r['id']:20s} {r['paper']} {r['sitting']} Q{r['q_num']} — {r['total_marks']} marks")

print("\nTechniques:")
for r in conn.execute("SELECT id, title FROM techniques"):
    print(f"  {r['id']}\n    -> {r['title']}")

print("\nTags in use:")
for r in conn.execute("SELECT DISTINCT tag_type, tag_value FROM question_tags ORDER BY tag_type, tag_value"):
    print(f"  [{r['tag_type']}] {r['tag_value']}")

Questions:
  P1_June2022_Q15      P1 June2022 Q15 — 10 marks
  P2_June2019_Q13      P2 June2019 Q13 — 10 marks

Techniques:
  eliminate-variable-using-constraint
    -> Eliminate a variable using the given constraint before differentiating
  differentiate-then-solve-stationary-point
    -> Differentiate the single-variable expression and set the derivative to zero
  second-derivative-test-for-max-min
    -> Use the second derivative to justify a maximum or minimum

Tags in use:
  [technique] differentiate-then-solve-stationary-point
  [technique] eliminate-variable-using-constraint
  [technique] second-derivative-test-for-max-min
  [topic] optimisation-with-derivatives


## 2. Look up one question in full

Change `QUESTION_ID` below to see a different question (see the list printed above).

In [3]:
QUESTION_ID = "P1_June2022_Q15"

q = conn.execute("SELECT * FROM questions WHERE id = ?", (QUESTION_ID,)).fetchone()

print(f"=== {q['id']} ({q['paper']} {q['sitting']}, {q['total_marks']} marks) ===\n")
print("QUESTION:")
print(q['question_text'])
print("\nMARK SCHEME:")
print(q['mark_scheme_text'])
print(f"\n(source: {q['source_qp_file']} / {q['source_ms_file']})")

=== P1_June2022_Q15 (P1 June2022, 10 marks) ===

QUESTION:
A company makes toys for children. Figure 5 shows the design for a solid toy shaped like a piece of cheese. The toy is modelled so that: face ABC is a sector of a circle with radius r cm and centre A; angle BAC = 0.8 radians; faces ABC and DEF are congruent; edges AD, CF and BE are perpendicular to faces ABC and DEF; edges AD, CF and BE have length h cm. Given that the volume of the toy is 240 cm^3, (a) show that the surface area of the toy, S cm^2, is given by S = 0.8r^2 + 1680/r, making your method clear. (4 marks) Using algebraic differentiation, (b) find the value of r for which S has a stationary point. (4 marks) (c) Prove, by further differentiation, that this value of r gives the minimum surface area of the toy. (2 marks)

MARK SCHEME:
(a) M1: Sets up an allowable equation using volume = 240, e.g. (1/2) r^2 * 0.8h = 240 => h = 600/r^2. A1: correct h. dM1: attempts to substitute their h into S = 2*(1/2 r^2 * 0.8) + 2rh + 

## 3. The retrieval step: topic tag → questions + techniques

This is the actual query the chatbot pipeline will run before calling an LLM: given a topic, pull every matching question and every technique note tagged for it. Try changing `TOPIC` once more topics exist.

In [4]:
def retrieve_for_topic(conn, topic: str):
    questions = conn.execute(
        """SELECT q.* FROM questions q
           JOIN question_tags qt ON qt.question_id = q.id
           WHERE qt.tag_type = 'topic' AND qt.tag_value = ?""",
        (topic,),
    ).fetchall()

    techniques = conn.execute(
        """SELECT DISTINCT t.* FROM techniques t
           JOIN question_tags qt ON qt.tag_value = t.id
           JOIN questions q ON q.id = qt.question_id
           JOIN question_tags topic_qt ON topic_qt.question_id = q.id
           WHERE qt.tag_type = 'technique'
             AND topic_qt.tag_type = 'topic' AND topic_qt.tag_value = ?""",
        (topic,),
    ).fetchall()

    return questions, techniques


TOPIC = "optimisation-with-derivatives"
questions, techniques = retrieve_for_topic(conn, TOPIC)

print(f"Topic: {TOPIC}")
print(f"\n{len(questions)} question(s):")
for q in questions:
    print(f"  - {q['id']}")

print(f"\n{len(techniques)} technique(s) used across those questions:")
for t in techniques:
    print(f"  - {t['title']}")

Topic: optimisation-with-derivatives

2 question(s):
  - P1_June2022_Q15
  - P2_June2019_Q13

3 technique(s) used across those questions:
  - Differentiate the single-variable expression and set the derivative to zero
  - Eliminate a variable using the given constraint before differentiating
  - Use the second derivative to justify a maximum or minimum


In [13]:
import pandas as pd
test_df = pd.read_sql_query("SELECT * FROM questions", conn)

In [15]:
test_df['question_text'].loc[0]

'A company makes toys for children. Figure 5 shows the design for a solid toy shaped like a piece of cheese. The toy is modelled so that: face ABC is a sector of a circle with radius r cm and centre A; angle BAC = 0.8 radians; faces ABC and DEF are congruent; edges AD, CF and BE are perpendicular to faces ABC and DEF; edges AD, CF and BE have length h cm. Given that the volume of the toy is 240 cm^3, (a) show that the surface area of the toy, S cm^2, is given by S = 0.8r^2 + 1680/r, making your method clear. (4 marks) Using algebraic differentiation, (b) find the value of r for which S has a stationary point. (4 marks) (c) Prove, by further differentiation, that this value of r gives the minimum surface area of the toy. (2 marks)'

In [17]:
question_tags_df = pd.read_sql_query("SELECT * FROM question_tags", conn)
question_tags_df

,question_id,tag_type,tag_value
0,P1_June2022_Q15,topic,optimisation-with-derivatives
1,P1_June2022_Q15,technique,eliminate-variable-using-constraint
2,P1_June2022_Q15,technique,differentiate-then-solve-stationary-point
3,P1_June2022_Q15,technique,second-derivative-test-for-max-min
4,P2_June2019_Q13,topic,optimisation-with-derivatives
5,P2_June2019_Q13,technique,eliminate-variable-using-constraint
6,P2_June2019_Q13,technique,differentiate-then-solve-stationary-point


In [18]:
technique_df = pd.read_sql_query("SELECT * FROM techniques", conn)
technique_df

,id,title,content,contrast_with
0,eliminate-variable-using-constraint,Eliminate a variable using the given constrain...,Optimisation problems in this topic usually gi...,
1,differentiate-then-solve-stationary-point,Differentiate the single-variable expression a...,Once the quantity to optimise is written as a ...,
2,second-derivative-test-for-max-min,Use the second derivative to justify a maximum...,Finding a stationary point only tells you the ...,


In [21]:
print(technique_df['title'].loc[0])
print(technique_df['content'].loc[0])

Eliminate a variable using the given constraint before differentiating
Optimisation problems in this topic usually give you a quantity to optimise (surface area, cost, etc.) that depends on two variables (e.g. r and h), plus a constraint (usually a fixed volume). You cannot differentiate a function of two variables with the tools available at this level, so the first step is always: use the constraint to write one variable in terms of the other, then substitute it into the expression for the quantity you're optimising. This turns a two-variable problem into a single-variable one that dy/dx methods can handle. Do this substitution algebraically and show every step — 'show that' parts award marks for the substitution itself, not just the final line.


## 4. "Recommend a similar question to try next"

For a given question, find other questions that share at least one technique tag — the same mechanic the real "try this next" recommendation feature will use, minus the embedding-based ranking we haven't built yet.

In [5]:
def similar_questions(conn, question_id: str):
    return conn.execute(
        """SELECT DISTINCT q.id, q.paper, q.sitting, q.q_num
           FROM questions q
           JOIN question_tags qt ON qt.question_id = q.id AND qt.tag_type = 'technique'
           WHERE qt.tag_value IN (
               SELECT tag_value FROM question_tags
               WHERE question_id = ? AND tag_type = 'technique'
           )
           AND q.id != ?""",
        (question_id, question_id),
    ).fetchall()


for r in similar_questions(conn, "P1_June2022_Q15"):
    print(f"  {r['id']} ({r['paper']} {r['sitting']} Q{r['q_num']})")

  P2_June2019_Q13 (P2 June2019 Q13)


## Scratch space

Use the cell below to try your own queries against `conn`.

In [6]:
# e.g. conn.execute("SELECT * FROM questions").fetchall()
